# 07 — Reading a Model as a Resource Budget

**Master syllabus coverage:** V2 1.7

## Why this matters

Before selecting a model or context length, an engineer should estimate whether weights, training state, activations, caches, workspaces, and runtime headroom plausibly fit the target device.

## Learning objectives

- Estimate weight, gradient, optimizer, activation-floor, and KV-cache memory.
- Track precision separately for each memory category.
- Compare full training, inference, LoRA, and grouped-query cache costs.
- State estimator assumptions and missing overhead explicitly.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Parameters are only the first memory category

Inference must store weights plus runtime buffers and KV cache. Training also stores
gradients, optimizer state, saved activations, and temporary workspaces. For ordinary
float32 AdamW, a rough persistent budget is $4P$ weight + $4P$ gradient + $8P$ moments
= $16P$ bytes, before activations and overhead.


In [ ]:
from dataclasses import dataclass

GIB = 2**30

@dataclass(frozen=True)
class ModelBudget:
    parameters: int
    bytes_per_weight: float
    layers: int
    context: int
    batch: int
    kv_heads: int
    head_dim: int
    bytes_per_cache_value: int = 2

    def weights_gib(self) -> float:
        return self.parameters * self.bytes_per_weight / GIB

    def adamw_persistent_gib(self) -> float:
        # Assumes fp32 weight, gradient, and two fp32 moment buffers.
        return self.parameters * 16 / GIB

    def kv_cache_gib(self) -> float:
        values = 2 * self.layers * self.batch * self.context * self.kv_heads * self.head_dim
        return values * self.bytes_per_cache_value / GIB


## 2. Precision changes weight storage, not every other category automatically

Quantized inference weights may use 8 or 4 bits plus scales/metadata. Activations and KV
cache often use a different dtype. Training may keep master weights or optimizer moments
in float32. State the precision of each category rather than saying “the model is FP16.”


In [ ]:
model_sizes = [10_000_000, 100_000_000, 500_000_000, 1_500_000_000]
precisions = {"FP32": 4, "FP16/BF16": 2, "INT8": 1, "INT4 ideal": 0.5}
for parameters in model_sizes:
    row = {name: parameters * bytes_per_weight / GIB for name, bytes_per_weight in precisions.items()}
    print(f"{parameters/1e6:6.0f}M params:", {key: f"{value:.2f} GiB" for key, value in row.items()})


## 3. KV cache grows with batch and context

Approximate decoder cache elements:

$$2LBT H_{kv}D$$

for keys and values. Grouped-query attention reduces $H_{kv}$ without changing the number
of query heads. Cache allocation policy, padding, fragmentation, and temporary attention
work add real overhead beyond this clean formula.


In [ ]:
for context in (512, 2_048, 8_192, 32_768):
    mha = ModelBudget(125_000_000, 2, 12, context, 1, 12, 64)
    gqa = ModelBudget(125_000_000, 2, 12, context, 1, 4, 64)
    print(f"T={context:5}: MHA cache={mha.kv_cache_gib():.3f} GiB, GQA cache={gqa.kv_cache_gib():.3f} GiB")


## 4. Activation memory depends on graph and implementation

Autograd saves values required by backward. A rough residual-sized estimate
`B×T×C×L×bytes` is only a floor: attention intermediates, MLP expansion, normalization,
dropout masks, temporary kernels, and allocator fragmentation add memory. Activation
checkpointing trades recomputation for fewer saved tensors.


In [ ]:
def residual_floor_gib(batch, context, width, layers, bytes_per_value=2):
    return batch * context * width * layers * bytes_per_value / GIB

for batch in (1, 8, 32):
    floor = residual_floor_gib(batch, 1024, 768, 12)
    print(f"batch={batch:2}: one residual-sized fp16 tensor/layer={floor:.3f} GiB")


## 5. LoRA changes trainable state, not the frozen base requirement

A rank-r adapter for `[out,in]` trains `r(out+in)` values instead of `out×in`, reducing
gradient and optimizer state for that update. The frozen base weights still occupy device
memory unless quantized or offloaded. QLoRA combines quantized frozen weights with trainable
low-rank adapters.


In [ ]:
def lora_parameters(input_width: int, output_width: int, rank: int) -> int:
    return rank * (input_width + output_width)

full = 4096 * 4096
for rank in (4, 8, 64):
    adapter = lora_parameters(4096, 4096, rank)
    print(f"rank={rank:2}: adapter={adapter:,}, full={full:,}, ratio={adapter/full:.4%}")


## 6. Feasibility includes headroom

Never compare a theoretical total directly with all reported device memory. Reserve space
for the OS/browser, runtime, allocator fragmentation, tokenizer/application, and workload
spikes. A useful estimator reports assumptions and ranges, then validates them with peak
memory measurements on the target runtime.


In [ ]:
example = ModelBudget(500_000_000, 2, 24, 4096, 1, 8, 128)
estimate = {
    "weights_gib": example.weights_gib(),
    "kv_cache_gib": example.kv_cache_gib(),
    "known_total_gib": example.weights_gib() + example.kv_cache_gib(),
    "not_included": ["quantization metadata", "activations", "workspaces", "fragmentation", "runtime"],
}
print(estimate)
assert estimate["known_total_gib"] > estimate["weights_gib"]


## Failure modes to recognize

- **Checkpoint-only budget:** runtime or training state causes out-of-memory.
- **One-dtype assumption:** weights, moments, activations, and cache are miscounted.
- **Context/batch omission:** KV cache and activations grow beyond the estimate.
- **No headroom:** allocator or application overhead consumes the final available memory.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Build a table for 10M–1.5B parameter models across FP32, FP16, INT8, and INT4 storage.
2. Add configurable SGD, AdamW, and 8-bit optimizer-state policies to the estimator.
3. Choose a target Mac/browser memory budget and defend one plausible Humor Machine configuration.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can estimate whether a model can load, train, or decode at a given context while naming every important excluded cost.

**Next:** 08 — Systems profiler capstone.
